# \[CDISC\] CDISC ADaM Clinical Trial Analysis

SEOYEON CHOI  
2026-03-03

# ADaM Analysis Data Model

> The **ADaM (Analysis Data Model)** dataset is a CDISC standard for
> organizing clinical trial data to support statistical analysis,
> regulatory reporting, and traceability. Derived from **SDTM (Study
> Data Tabulation Model)** data, ADaM datasets are designed to be
> **“analysis-ready”** for generating tables, listings, and figures.
> They are required by **regulatory agencies like the FDA and PMDA**.

# Reference

-   [Phase II Alzheimer’s clinical trial
    Data](https://github.com/cdisc-org/sdtm-adam-pilot-project/tree/master)
    -   [adamdata
        guide](https://github.com/cdisc-org/sdtm-adam-pilot-project/blob/master/updated-pilot-submission-package/900172/m5/datasets/cdiscpilot01/analysis/adam/datasets/dataguide.pdf)
-   [Explanation](https://www.lexjansen.com/pharmasug/2012/DS/PharmaSUG-2012-DS18.pdf)

**Methodology**

> This was a prospective, randomized, multi-center, double-blind,
> placebo-controlled, parallel-group study. Subjects were randomized
> equally to placebo, xanomeline low dose, or xanomeline

**Study Overview**

-   Study design:
    -   Phase II randomized clinical trial
    -   3 treatment arms (Placebo / Low dose / High dose)
    -   Duration: 26 weeks
    -   Population: Mild–moderate Alzheimer’s disease patients
-   Primary objectives
    -   Evaluate efficacy of active drug vs placebo
    -   Assess safety and tolerability
-   Endpoints
    -   Primary efficacy:
        -   ADAS-Cog score
    -   Secondary:
        -   CIBIC+
        -   NPI
    -   Safety
        -   Adverse events
        -   Laboratory tests
        -   Vital signs

# Import

In [353]:
import pandas as pd
import numpy as np
import pyreadstat
import os
from pathlib import Path

from scipy import stats


# Data

In [3]:
data_folder = Path("../../../../delete/adam/")

files = [
    "adae.xpt",
    "adlbc.xpt",
    "adlbh.xpt",
    "adlbhy.xpt",
    "adqsadas.xpt",
    "adqscibc.xpt",
    "adqsnpix.xpt",
    "adsl.xpt",
    "adtte.xpt",
    "advs.xpt"
]

In [4]:
data = {}
metadata = {}

In [5]:
for f in files:
    full_path = data_folder / f
    df, meta = pyreadstat.read_xport(full_path)
    
    key = f.replace(".xpt","")
    
    data[key] = df
    metadata[key] = meta
    
    print(f"{f} loaded:", df.shape)

adae.xpt loaded: (1191, 55)
adlbc.xpt loaded: (74264, 46)
adlbh.xpt loaded: (49932, 46)
adlbhy.xpt loaded: (9954, 43)
adqsadas.xpt loaded: (12463, 40)
adqscibc.xpt loaded: (730, 36)
adqsnpix.xpt loaded: (31140, 41)
adsl.xpt loaded: (254, 48)
adtte.xpt loaded: (254, 26)
advs.xpt loaded: (32139, 34)

# Summary

## Analysis Populations

-   Population Summary

In [342]:
df = data['adsl']
rows = ['SAFFL','ITTFL','EFFFL']

result = []

for flag in rows:

    ct = pd.crosstab(df[flag], df['ARM']).reindex(['Y','N'], fill_value=0)
    
    pct = ct.div(ct.sum(axis=0), axis=1) * 100
    
    formatted = ct.astype(str) + " (" + pct.round(1).astype(str) + "%)"
    
    total_ct = df[flag].value_counts().reindex(['Y','N'], fill_value=0)
    total_pct = total_ct / len(df) * 100
    
    formatted['Total'] = (
        total_ct.astype(str) + " (" +
        total_pct.round(0).astype(int).astype(str) + "%)"
    )
    
    formatted.index = [f"{flag}={i}" for i in formatted.index]
    
    result.append(formatted)

table = pd.concat(result)

table

## Demographics and Baseline Characteristics

-   Describe comparability of treatment groups

In [365]:
def _fmt_p(p):
    if pd.isna(p):
        return ""
    if p < 0.001:
        return "<0.001"
    return f"{p:.3f}"


def _mean_sd(x):
    x = pd.to_numeric(x, errors="coerce").dropna()
    if len(x) == 0:
        return ""
    return f"{x.mean():.1f} ({x.std(ddof=1):.1f})"


def _n_pct(n, denom):
    if denom == 0:
        return f"{n} (0.0%)"
    return f"{n} ({100*n/denom:.1f}%)"


def _shapiro_p(x):
    x = pd.to_numeric(x, errors="coerce").dropna()
    n = len(x)
    if n < 3:
        return np.nan
    if n > 5000:
        x = x.sample(5000, random_state=1)
    try:
        return stats.shapiro(x).pvalue
    except Exception:
        return np.nan


def _cont_pvalue(df, var, arm_col, normality=True, alpha=0.05):
    groups = []
    for _, sub in df.groupby(arm_col, dropna=False):
        x = pd.to_numeric(sub[var], errors="coerce").dropna().values
        groups.append(x)

    nonempty = [g for g in groups if len(g) > 0]
    if len(nonempty) < 2:
        return (np.nan, "NA")

    if normality:
        ps = []
        for _, sub in df.groupby(arm_col, dropna=False):
            ps.append(_shapiro_p(sub[var]))

        # any NaN (too small n) or any p<=alpha -> Kruskal (conservative)
        if any(pd.isna(p) for p in ps) or any(p <= alpha for p in ps):
            try:
                return (stats.kruskal(*nonempty).pvalue, "Kruskal–Wallis")
            except Exception:
                return (np.nan, "Kruskal–Wallis")
        else:
            try:
                return (stats.f_oneway(*nonempty).pvalue, "ANOVA")
            except Exception:
                return (np.nan, "ANOVA")
    else:
        try:
            return (stats.f_oneway(*nonempty).pvalue, "ANOVA")
        except Exception:
            return (np.nan, "ANOVA")


def _cat_pvalue(df, var, arm_col):
    tab = pd.crosstab(df[var], df[arm_col], dropna=False)

    if tab.shape[0] < 2 or tab.shape[1] < 2:
        return (np.nan, "NA")

    try:
        chi2, p, dof, expected = stats.chi2_contingency(tab.values, correction=False)
    except Exception:
        return (np.nan, "Chi-square")

    # 2x2 and any expected <5 -> Fisher
    if tab.shape == (2, 2) and (expected < 5).any():
        try:
            _, p_f = stats.fisher_exact(tab.values)
            return (p_f, "Fisher’s exact")
        except Exception:
            return (p, "Chi-square")

    return (p, "Chi-square")


def make_table1_adsl(
    df,
    arm_col="ARM",
    continuous=None,
    categorical=None,
    labels=None,
    normality_for_continuous=True,
    include_missing_row=True,
    pct_decimals=1,
    cont_decimals=1,
):
    continuous = continuous or []
    categorical = categorical or []
    labels = labels or {}

    # ARM levels in appearance order (keep NaN last)
    arm_levels = df[arm_col].dropna().unique().tolist()
    if df[arm_col].isna().any():
        arm_levels = arm_levels + [np.nan]

    total_n = len(df)

    # Footnote letters by test
    test_to_letter = {}
    letters = list("123456789")

    def _letter_for(test_name):
        if test_name in ("", "NA", None):
            return ""
        if test_name not in test_to_letter:
            test_to_letter[test_name] = letters[len(test_to_letter)]
        return test_to_letter[test_name]

    def _get_arm_subset(arm):
        if pd.isna(arm):
            return df[df[arm_col].isna()]
        return df[df[arm_col] == arm]

    def _n_pct_fmt(n, denom):
        if denom == 0:
            return f"{n} (0.{ '0'*pct_decimals }%)" if pct_decimals > 0 else f"{n} (0%)"
        fmt = f"{{:.{pct_decimals}f}}"
        return f"{n} ({fmt.format(100*n/denom)}%)"

    def _mean_sd_fmt(x):
        x = pd.to_numeric(x, errors="coerce").dropna()
        if len(x) == 0:
            return ""
        fmt = f"{{:.{cont_decimals}f}}"
        return f"{fmt.format(x.mean())} ({fmt.format(x.std(ddof=1))})"

    rows = []
    index = []

    def _add_row(row_label, values_by_arm, pval=None, test_name=None):
        row = {}
        for arm in arm_levels:
            row[arm] = values_by_arm.get(arm, "")
        row["Total"] = values_by_arm.get("Total", "")
        if pval is None or test_name in ("", "NA", None):
            row["p-value"] = ""
        else:
            lt = _letter_for(test_name)
            row["p-value"] = f"{_fmt_p(pval)}[{lt}]"
        rows.append(row)
        index.append(row_label)

    # --------- CATEGORICAL ---------
    for var in categorical:
        var_label = labels.get(var, var)

        p, test = _cat_pvalue(df, var, arm_col)

        # header row (blank cells + p-value)
        empty_vals = {arm: "" for arm in arm_levels}
        empty_vals["Total"] = ""
        _add_row(var_label, empty_vals, p, test)

        levels = pd.Series(df[var].dropna().unique()).tolist()
        for lvl in levels:
            vals = {}
            for arm in arm_levels:
                sub = _get_arm_subset(arm)
                n = (sub[var] == lvl).sum()
                denom = len(sub)
                vals[arm] = _n_pct_fmt(n, denom)
            vals["Total"] = _n_pct_fmt((df[var] == lvl).sum(), total_n)
            _add_row(f"  {lvl}", vals)

        if include_missing_row:
            vals = {}
            for arm in arm_levels:
                sub = _get_arm_subset(arm)
                n_miss = sub[var].isna().sum()
                denom = len(sub)
                vals[arm] = _n_pct_fmt(n_miss, denom)
            vals["Total"] = _n_pct_fmt(df[var].isna().sum(), total_n)
            # _add_row("  Missing", vals)

    # --------- CONTINUOUS ---------
    for var in continuous:
        var_label = labels.get(var, var)

        vals = {}
        for arm in arm_levels:
            sub = _get_arm_subset(arm)
            vals[arm] = _mean_sd_fmt(sub[var])
        vals["Total"] = _mean_sd_fmt(df[var])

        p, test = _cont_pvalue(df, var, arm_col, normality=normality_for_continuous)
        _add_row(var_label, vals, p, test)

        if include_missing_row:
            vals_m = {}
            for arm in arm_levels:
                sub = _get_arm_subset(arm)
                n_miss = pd.to_numeric(sub[var], errors="coerce").isna().sum()
                denom = len(sub)
                vals_m[arm] = _n_pct_fmt(n_miss, denom)
            vals_m["Total"] = _n_pct_fmt(pd.to_numeric(df[var], errors="coerce").isna().sum(), total_n)
            # _add_row("  Missing", vals_m)

    out = pd.DataFrame(rows, index=index)

    # Rename NaN ARM column if exists
    col_rename = {}
    for arm in arm_levels:
        if pd.isna(arm):
            col_rename[arm] = "Missing ARM"
        else:
            col_rename[arm] = str(arm)
    out = out.rename(columns=col_rename)

    # Footnotes
    letter_to_test = {v: k for k, v in test_to_letter.items()}
    footnotes = [f"{lt} {letter_to_test[lt]}" for lt in sorted(letter_to_test.keys())]

    return out, footnotes


# =========================
# Usage
# =========================
df = data["adsl"]

labels = {
    "SEX": "Sex",
    "AGE": "Age",
    "RACE": "Race",
    "ETHNIC": "Ethnicity",
    "BMIBL": "Baseline BMI (kg/m^2)",
    "HEIGHTBL": "Baseline Height (cm)",
    "WEIGHTBL": "Baseline Weight (kg)",
    "EDUCLVL": "Years of Education",
}

categorical = ["SEX", "RACE", "ETHNIC"]
continuous = ["AGE", "BMIBL", "HEIGHTBL", "WEIGHTBL", "EDUCLVL"]

table1, footnotes = make_table1_adsl(
    df,
    arm_col="ARM",
    continuous=continuous,
    categorical=categorical,
    labels=labels,
    normality_for_continuous=False,
    include_missing_row=True,
    pct_decimals=0, 
    cont_decimals=0,
)

display(table1)

print("Footnotes:")
for f in footnotes:
    print(f)

Footnotes:
1 Chi-square
2 ANOVA

## Treatment Exposure

-   exposure duration
-   compliance

‘TRT01P’: ‘Planned Treatment for Period 01’, ‘TRT01PN’: ‘Planned
Treatment for Period 01 (N)’, ‘TRT01A’: ‘Actual Treatment for Period
01’, ‘TRT01AN’: ‘Actual Treatment for Period 01 (N)’, ‘TRTSDT’: ‘Date of
First Exposure to Treatment’, ‘TRTEDT’: ‘Date of Last Exposure to
Treatment’, ‘TRTDUR’: ‘Duration of Treatment (days)’,

-   Efficacy Analysis
    -   mean change from baseline
    -   ANCOVA / MMRM
    -   treatment comparison
-   Primary endpoint
    -   adas-cog change from baseline

In [328]:
data['adqsadas']

In [329]:
data['adqscibc']

-   Safety Analysis
    -   TEAE
    -   serious AE
    -   AE leading to discontinuation

In [330]:
data['adae']

-   Laboratory Analysis
    -   mean chang from baseline
    -   shift tables

In [332]:
data['adlbc']

In [333]:
data['adlbh']

-   Vital Signs
    -   systolic BP
    -   diastolic BP
    -   heart rate
    -   weight

In [334]:
data['advs']

-   Time-to-event Analysis
    -   Kaplan-Meier curve
    -   median survival
    -   Cox model

In [335]:
data['adtte']

-   Demographics

-   Table 14.1

    -   Demographic and Baseline Characteristics

In [ ]:
Columns
Placebo
Low Dose
High Dose
Total

Rows

Age mean (SD)
Sex n (%)
Race n (%)
Baseline ADAS score
Weight
Disease duration

-   Disposition
-   Table 14.2
    -   Patient Disposition

Randomized Treated Completed Discontinued

In [ ]:
Exposure
Table 14.3

Treatment Exposure Summary

Treatment duration
Mean exposure
Dose interruptions

Primary efficacy Table 14.4

Change from Baseline in ADAS-Cog

Baseline Week 12 Week 26 Mean change Treatment difference 95% CI p-value

In [ ]:
Secondary efficacy
Table 14.5

CIBIC+ Score

Improved
No change
Worsened

In [ ]:
Safety tables
Table 14.6

Patients with ≥1 TEAE

Placebo
Low dose
High dose

In [ ]:
Table 14.7

TEAE by System Organ Class

Example

Cardiac disorders
GI disorders
Nervous system disorders
Psychiatric disorders

In [ ]:
Table 14.8

Top Preferred Terms

Headache
Nausea
Fatigue
Dizziness

In [ ]:
Laboratory
Table 14.9

Laboratory Parameters – Change from Baseline

ALT
AST
Creatinine
Hemoglobin

In [ ]:
Table 14.10

Laboratory Shift Table

Low → Normal
Normal → High
High → Normal

In [ ]:
Vital signs
Table 14.11

Vital Signs Summary

Systolic BP
Diastolic BP
Heart rate
Weight

In [ ]:
Figures
Figure 14.1

ADAS-Cog change from baseline over time

(line plot)

In [ ]:
Figure 14.2

Kaplan-Meier survival curve

(dataset: ADTTE)

In [ ]:
Figure 14.3

AE incidence bar chart

## ADSL

> The first is ADSL (Subject Level Analysis Dataset), a
> one-recordper-subject structure that contains subject-level
> attributes. Because of its structure, it can be merged onto any other
> clinical dataset, including other ADaM datasets and SDTM datasets.

In [6]:
data['adsl'].head()

In [7]:
metadata['adsl'].column_names_to_labels

# Table

## Table 11-1. Demographic Characteristics

In [289]:
data2 = data['adsl'].copy()

data2['RACE2'] = data2['RACE'].replace({
    'WHITE': 'White/Caucasian',
    'BLACK OR AFRICAN AMERICAN': 'Other',
    'AMERICAN INDIAN OR ALASKA NATIVE': 'Other'
})

In [290]:
n_counts = data['adsl'].groupby('ARM')['USUBJID'].nunique()
total_n = data['adsl']['USUBJID'].nunique()
n_counts['Total'] = total_n

In [316]:
def mean_min_max_table(df, var, group_col='ARM', total_label='Total',
                       mean_digits=1, minmax_digits=0):

    arm_summary = (
        df.groupby(group_col)[var]
          .agg(['mean', 'min', 'max'])
    )

    total_summary = (
        df[var]
          .agg(['mean', 'min', 'max'])
          .to_frame().T
    )
    total_summary.index = [total_label]

    final = pd.concat([arm_summary, total_summary])

    final[f'{var} Mean (Min–Max)'] = (
        final['mean'].round(mean_digits).astype(str)
        + " ("
        + final['min'].round(minmax_digits).astype(int).astype(str)
        + "–"
        + final['max'].round(minmax_digits).astype(int).astype(str)
        + ")"
    )

    return final[[f'{var} Mean (Min–Max)']]

In [ ]:
def categorical_table(df, var, group_col='ARM', total_label='Total', pct_digits=0):

    ct = pd.crosstab(df[var], df[group_col])

    pct = ct.div(ct.sum(axis=0), axis=1) * 100

    formatted = (
        ct.astype(str) + " (" +
        pct.round(pct_digits).astype(int).astype(str) + "%)"
    )

    total_n = df.shape[0]
    total_ct = df[var].value_counts()
    total_pct = total_ct / total_n * 100

    formatted[total_label] = (
        total_ct.astype(str) + " (" +
        total_pct.round(pct_digits).astype(int).astype(str) + "%)"
    )

    formatted.columns.name = None

    return formatted

In [313]:
Table11_1 = pd.concat([mean_min_max_table(data['adsl'],'AGE').T,
                    categorical_table(data['adsl'],'SEX').reset_index().set_index('SEX').reindex(['M', 'F']),
                    categorical_table(data2,'RACE2').reset_index().set_index('RACE2').reindex(['White/Caucasian', 'Other']),
                    mean_min_max_table(data['adsl'],'EDUCLVL').T,
                   ])

In [314]:
Table11_1.columns = [
    f"{col} (n={n_counts[col]})" if col in n_counts.index
    else col
    for col in final2.columns
]

In [315]:
Table11_1

In [319]:
data['adtte']